## LoadBalancer

In this notebook, we look at Services of type LoadBalancer in Kubernetes.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool on your computer.

## Installing a LoadBalancer

A LoadBalancer is a Kubernetes Service that exposes an *internal* Kubernetes application to the *outside world* (i.e. the *internet* when using a cloud provider) and also functions as a Layer 4 load balancer.

- When using a Minikube Kubernetes cluster, you can enable the MetalLB LoadBalancer addon.

**Note**
- The MetalLB load balancer can also be installed in a Kubernetes cluster using Helm.
  
  ```bash
  helm repo add metallb https://metallb.github.io/metallb
  helm repo update
  helm upgrade --install metallb metallb/metallb --create-namespace --namespace metallb-system
  ```

In [1]:
!minikube addons enable metallb

❗  metallb is a 3rd party addon and is not maintained or verified by minikube maintainers, enable at your own risk.
❗  metallb does not currently have an associated maintainer.
    ▪ Using image quay.io/metallb/speaker:v0.9.6
    ▪ Using image quay.io/metallb/controller:v0.9.6
🌟  The 'metallb' addon is enabled


- List the MetalLB Pods.
  - Notice the subnet (e.g. `192.168.49.x`) used by the `speaker` Pod.

In [3]:
!kubectl get pods -n metallb-system -o wide

NAME                         READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
controller-cff57fcb5-9xhzz   1/1     Running   0          27s   10.244.0.29    minikube   <none>           <none>
speaker-hsbv2                1/1     Running   0          27s   192.168.49.2   minikube   <none>           <none>


- Deploy a ConfigMap
  - Open the file `manifests/configmap.yaml`.
  - Change the last row so that it matches your subnet.
    - E.g. if your `speaker` nodes are on subnet `192.168.49.x`, change the last row to `192.168.49.100-192.168.49.110`
    - This let's MetalLB offer `EXTERNAL-IP` addresses in that range.
  - Save the YAML file.
- Run the cell below to deploy the ConfigMap (which configures MetalLB).

In [4]:
!kubectl apply -f manifests/configmap.yaml

configmap/config configured


## List Nodes

- Since we are using a single-node Minikube Kubernetes cluster, there is only one node.

In [5]:
!kubectl get nodes -o wide

NAME       STATUS   ROLES           AGE   VERSION   INTERNAL-IP    EXTERNAL-IP   OS-IMAGE                         KERNEL-VERSION      CONTAINER-RUNTIME
minikube   Ready    control-plane   86m   v1.35.1   192.168.49.2   <none>        Debian GNU/Linux 12 (bookworm)   6.17.0-14-generic   docker://29.2.1


## Deploy a Deployment

- The Deployment definition is in the YAML file `manifests/deploy-app.yaml`.

In [6]:
!kubectl apply -f manifests/deploy-app.yaml

deployment.apps/deploy-nginx created


## Let's look at the Deployment's YAML

**Note:**

- The number of `replicas` is set to 3.
- The `revisionHistoryLimit` is set to 3.
- The Pod template's container is listening on `containerPort` 80.
- The Pod template's `labels` are `app: nginx` and `env: prod`
  - The Deployment's `matchLabels` match these labels
    - Therefore, the Deployment will create 3 replicas from the Pod template.
  - The Service's `selector` matches these labels
    - Therefore, the Service will load balance between the three Pod replicas.

```bash
apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-nginx         # the deployment is named deploy-nginx
spec:
  replicas: 3                 # the deployment defines three replicas
  revisionHistoryLimit: 3     # the deployment defines a maximum of three ReplicaSet revisions to store in history
  selector:
    matchLabels:
      app: nginx              # the deployment has two matchLabels that match the Pod template's labels below
      env: prod               # app = nginx and env = prod
  template:
    metadata:
      name: myapp-pod
      labels:
        app: nginx             # the Pod template has two labels defined (the Service's selector matches these two labels):
        env: prod              # app = nginx and env = prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine    # the container is based on the nginx:alpine image
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi    
        ports:
        - containerPort: 80    # the Pod template's container is listening on containerPort 80
```

In [7]:
#!type manifests\deploy-app.yaml # use this on Windows
!cat manifests/deploy-app.yaml # use this on Linux/Mac

apiVersion: apps/v1
kind: Deployment
metadata:
  name: deploy-nginx
  labels:
    app: nginx
    env: prod
spec:
  replicas: 3
  revisionHistoryLimit: 3
  selector:
    matchLabels:
      app: nginx
      env: prod
  template:
    metadata:
      labels:
        app: nginx
        env: prod
    spec:
      containers:
      - name: nginx
        image: nginx:alpine
        ports:
        - containerPort: 80
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi

## Deploy a Service

- The Service definition is in the YAML file `manifests/loadbalancer.yaml`.

In [8]:
!kubectl apply -f manifests/loadbalancer.yaml

service/svc-example created


## Let's look at the Service's YAML

**Note:**

- The service's name is `svc-example`.
- The service's is:
  - Of type `LoadBalancer` which means it will be assigned an external (public) IP address.
  - Listening on `port` 8080.
  - Redirecting traffic to `targetPort` 80.
- The servie's `selector` defines two labels (that match the Pod template's labels for the Deployment):
  - `app: nginx` and `env: prod`
  - This means the Service will (round robin) load balance between the three Pods in the Deployment's ReplicaSet.

```bash
apiVersion: v1
kind: Service
metadata:
  name: svc-example    # the Service's name is svc-example
spec:
  type: LoadBalancer   # the Service's type is LoadBalancer
  selector:
    app: nginx         # the Service's selector defines two labels (the Service's selector matches the Pod template's two labels):
    env: prod          # app = nginx and env = prod
  ports:
  - protocol: TCP      # the Service's protocol is TCP
    port: 8080         # the Service is listening for internal (private) traffic on port 8080
    targetPort: 80     # the Service is redirecting traffic to targetPort 80
```

In [9]:
#!type manifests\loadbalancer.yaml # use this on Windows
!cat manifests/loadbalancer.yaml # use this on Linux/Mac

apiVersion: v1
kind: Service
metadata:
  name: svc-example
spec:
  type: LoadBalancer
  selector:
    app: nginx
    env: prod
  ports:
  - protocol: TCP
    port: 8080
    targetPort: 80

## Get Pods

- We see that 3 Pods are running.

In [10]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                           READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
deploy-nginx-6f689b949-bwb2m   1/1     Running   0          21s   10.244.0.32   minikube   <none>           <none>
deploy-nginx-6f689b949-dmplx   1/1     Running   0          21s   10.244.0.30   minikube   <none>           <none>
deploy-nginx-6f689b949-jpl5r   1/1     Running   0          21s   10.244.0.31   minikube   <none>           <none>


## Get Services

- Notice the `TYPE` is `LoadBalancer` for the `svc-example` Service.
- Notice the `EXTERNAL-IP` and `PORT` 8080 used by the `Loadbalancer`.
  - In Docker Desktop's Kubernetes cluster, `EXTERNAL-IP` will be `localhost`.
  - In Minikube's Kubernetes cluster, `EXTERNAL-IP` will be an IP addess on your `speaker` nodes' subnet (that you configure using the ConfigMap above).
    - If the `EXTERNAL-IP` for your `LoadBalancer` is `<pending>`, you didn't install the MetalLB addon and/or didn't configure it correctly with the ConfigMap.
  - In a cloud provider's Kubernetes cluster, `EXTERNAL-IP` would be a proper external (public) IP Address.

In [11]:
#!kubectl get svc -o wide
!kubectl get services -o wide

NAME          TYPE           CLUSTER-IP       EXTERNAL-IP      PORT(S)          AGE   SELECTOR
kubernetes    ClusterIP      10.96.0.1        <none>           443/TCP          87m   <none>
svc-example   LoadBalancer   10.107.211.188   192.168.49.100   8080:30270/TCP   25s   app=nginx,env=prod


## Access the Load Balancer Service

- Use the Load Balancer Service's assigned `EXTERNAL-IP` address above and the Service's port number 8080: `http://EXTERNAL-IP:8080`
  - In Minikube's Kubernetes cluster, `EXTERNAL-IP` is the IP address you see under `EXTERNAL-IP` for your `LoadBalancer` above.
    - E.g. if your `EXTERNAL-IP` for your `LoadBalancer` above is `192.168.49.100`, you would use `http://192.168.49.100:8080`
  - When using a cloud provider, you would use the cloud provider's LoadBalancer `External-IP` address and your Service's port number.

In [12]:
!curl http://192.168.49.100:8080

<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
html { color-scheme: light dark; }
body { width: 35em; margin: 0 auto;
font-family: Tahoma, Verdana, Arial, sans-serif; }
</style>
</head>
<body>
<h1>Welcome to nginx!</h1>
<p>If you see this page, nginx is successfully installed and working.
Further configuration is required for the web server, reverse proxy, 
API gateway, load balancer, content cache, or other features.</p>

<p>For online documentation and support please refer to
<a href="https://nginx.org/">nginx.org</a>.<br/>
To engage with the community please visit
<a href="https://community.nginx.org/">community.nginx.org</a>.<br/>
For enterprise grade support, professional services, additional 
security features and capabilities please refer to
<a href="https://f5.com/nginx">f5.com/nginx</a>.</p>

<p><em>Thank you for using nginx.</em></p>
</body>
</html>


## Delete the Service and the Deployment

In [13]:
!kubectl delete -f manifests/loadbalancer.yaml
!kubectl delete -f manifests/deploy-app.yaml

service "svc-example" deleted from default namespace
deployment.apps "deploy-nginx" deleted from default namespace


## List Services, Deployments and Pods

- We see that the Service and Deployment with associated Pods were deleted.
  - The service with **NAME** `kubernetes` and **TYPE** `ClusterIP` is for routing requests to the Kubernetes API Server, and should never be deleted.

In [14]:
#!kubectl get svc
#!kubectl get deploy
#!kubectl get po
!kubectl get services
!kubectl get deployments
!kubectl get pods

NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   89m
No resources found in default namespace.
No resources found in default namespace.
